In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install pathlib
%pip install scikit-learn
%pip install imbalanced-learn
%pip install scipy
%pip install xgboost

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [1]:
import os
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"La máquina tiene {num_cores} núcleos disponibles.")
hilos_optimos = str(min(num_cores, 64)) 
print(f"Configurando OpenBLAS para usar {hilos_optimos} hilos máximos...")
os.environ['OPENBLAS_NUM_THREADS'] = hilos_optimos
os.environ['MKL_NUM_THREADS'] = hilos_optimos
os.environ['OMP_NUM_THREADS'] = hilos_optimos

La máquina tiene 224 núcleos disponibles.
Configurando OpenBLAS para usar 64 hilos máximos...


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import numpy as np
import pandas as pd
import itertools
import json
from pathlib import Path
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, recall_score,classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.combine import SMOTEENN


In [3]:
X = pd.read_pickle("Sets_Xy/X.pkl")
y = pd.read_pickle("Sets_Xy/y.pkl")

In [4]:
from sklearn.model_selection import train_test_split

#Division estratificada para muestras de cada clase a nivel de cada subset

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, 
    random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, 
    random_state=42)

In [5]:
#Encoding de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

mapeo_labels = pd.DataFrame({
    "label_original": le.classes_,
    "label_encoded": range(len(le.classes_))
})
print("Mapeo de etiquetas:\n", mapeo_labels)
class_names = le.classes_

Mapeo de etiquetas:
                label_original  label_encoded
0                      BENIGN              0
1                         Bot              1
2                        DDoS              2
3               DoS GoldenEye              3
4                    DoS Hulk              4
5            DoS Slowhttptest              5
6               DoS slowloris              6
7                 FTP-Patator              7
8                  Heartbleed              8
9                Infiltration              9
10                   PortScan             10
11                SSH-Patator             11
12    Web Attack  Brute Force             12
13  Web Attack  Sql Injection             13
14            Web Attack  XSS             14


In [6]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, method='tree', k=50, class_weight='balanced', random_state=42):
        self.method = method
        self.k = k
        self.class_weight = class_weight
        self.random_state = random_state
        self.selector = None
        self.feature_names_ = None
        self.support_mask_ = None

    def fit(self, X, y):
        self.feature_names_ = X.columns.tolist() if hasattr(X, 'columns') else list(range(X.shape[1]))
        
        X_array = X.values if hasattr(X, 'values') else X
        y_array = y.values if hasattr(y, 'values') else y
        
        total_features = X_array.shape[1]
        actual_k = min(self.k, total_features)
        
        if self.method == 'none':
            self.support_mask_ = np.ones(total_features, dtype=bool)
            return self

        if self.method == 'f_classif':
            self.selector = SelectKBest(score_func=f_classif, k=actual_k)
            self.selector.fit(X_array, y_array)
            self.support_mask_ = self.selector.get_support()

        elif self.method == 'tree':
            estimator = ExtraTreesClassifier(
                n_estimators=100,               
                max_depth=20,                   
                class_weight=self.class_weight,
                criterion='gini',               
                random_state=self.random_state, 
                n_jobs=-1
            )
            estimator.fit(X_array, y_array)
            importances = estimator.feature_importances_
            
            top_k_indices = np.argsort(importances)[-actual_k:]
            
            self.support_mask_ = np.zeros(total_features, dtype=bool)
            self.support_mask_[top_k_indices] = True

        return self

    def transform(self, X):
        if self.method == 'none':
            return X
            
        X_array = X.values if hasattr(X, 'values') else X
        X_transformed = X_array[:, self.support_mask_]

        if hasattr(X, 'columns'):
            selected_features = np.array(self.feature_names_)[self.support_mask_].tolist()
            return pd.DataFrame(X_transformed, columns=selected_features, index=X.index)
        else:
            return X_transformed

In [7]:
import joblib

X_train_none_2 = joblib.load("Sets_Oversampling_2/X_train_none.joblib")
y_train_none_2 = joblib.load("Sets_Oversampling_2/y_train_none.joblib")

X_train_smote_2 = joblib.load("Sets_Oversampling_2/X_train_smote.joblib")
y_train_smote_2 = joblib.load("Sets_Oversampling_2/y_train_smote.joblib")

X_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/X_train_smote_enn.joblib")
y_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/y_train_smote_enn.joblib")

X_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/X_train_smote_tomek.joblib")
y_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/y_train_smote_tomek.joblib")

X_val = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian.pkl")
y_val = pd.DataFrame(y_val)

X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")
y_test = pd.DataFrame(y_test)


In [8]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

def balanced_class_weight(y):
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

def polynomial_class_weight(y, alpha=0.25):
    classes, counts = np.unique(y, return_counts=True)
    weights = 1.0 / np.power(counts, alpha)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def log_ratio_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    total = np.sum(counts)
    weights = np.log(total / counts)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def effective_samples_class_weight(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def cost_sensitive_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    max_count = np.max(counts)
    weights = max_count / counts
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def focal_class_weight_improved(y, gamma=2.0):

    classes, counts = np.unique(y, return_counts=True)
    probs = counts / np.sum(counts)
    weights = (1 - probs) ** gamma / probs
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

In [9]:
train_datasets = {
    'none': (X_train_none_2, y_train_none_2.values.ravel() if isinstance(y_train_none_2, pd.DataFrame) else y_train_none_2.ravel()),
    'smote': (X_train_smote_2, y_train_smote_2.values.ravel() if isinstance(y_train_smote_2, pd.DataFrame) else y_train_smote_2.ravel()),
    'smote_enn': (X_train_smote_enn_2, y_train_smote_enn_2.values.ravel() if isinstance(y_train_smote_enn_2, pd.DataFrame) else y_train_smote_enn_2.ravel()),
    'smote_tomek': (X_train_smote_tomek_2, y_train_smote_tomek_2.values.ravel() if isinstance(y_train_smote_tomek_2, pd.DataFrame) else y_train_smote_tomek_2.ravel())
}

y_val_1d = y_val.values.ravel() if isinstance(y_val, pd.DataFrame) else y_val.ravel()
y_test_1d = y_test.values.ravel() if isinstance(y_test, pd.DataFrame) else y_test.ravel()

In [10]:
import joblib
X_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_none.joblib")
y_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_none.joblib")

X_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote.joblib")
y_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote.joblib")

X_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_enn.joblib")
y_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_enn.joblib")

X_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_tomek.joblib")
y_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_tomek.joblib")

X_val_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian_grouped.pkl")
X_test_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian_grouped.pkl")

y_val_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_val_grouped.pkl")
y_test_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_test_grouped.pkl")

In [11]:
train_datasets_grouped = {
    'none': (X_train_none_grouped, y_train_none_grouped.values.ravel() if isinstance(y_train_none_grouped, pd.DataFrame) else y_train_none_grouped.ravel()),
    'smote': (X_train_smote_grouped, y_train_smote_grouped.values.ravel() if isinstance(y_train_smote_grouped, pd.DataFrame) else y_train_smote_grouped.ravel()),
    'smote_enn': (X_train_smote_enn_grouped, y_train_smote_enn_grouped.values.ravel() if isinstance(y_train_smote_enn_grouped, pd.DataFrame) else y_train_smote_enn_grouped.ravel()),
    'smote_tomek': (X_train_smote_tomek_grouped, y_train_smote_tomek_grouped.values.ravel() if isinstance(y_train_smote_tomek_grouped, pd.DataFrame) else y_train_smote_tomek_grouped.ravel())
}

In [12]:
def save_confusion_matrix_xgb(log_folder, y_true, y_pred, trial_number, dataset_name, phase="val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'confusion matrix - {dataset_name} trial {trial_number} ({phase})')
    plt.ylabel('true label')
    plt.xlabel('predicted label')
    plt.savefig(f'{log_folder}/{dataset_name}_trial_{trial_number}_xgb_{phase}_cm.png', bbox_inches='tight')
    plt.close()

In [ ]:
import os
import optuna
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")
dir_name = "Logs_XGBoost_Baseline_1"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix_xgb(y_true, y_pred, trial_number, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'XGBoost Confusion Matrix - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/Trial_{trial_number}_XGB_CM.png')
    plt.close()

def objective_xgb(trial):
    x_train_raw, y_train_raw = train_datasets['none']
    
    xgb_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 6, 30), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),    
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.7, 1.0),
        'tree_method': 'hist', 
        'device': 'cuda',
        'objective': 'multi:softmax',
        'num_class': 15,
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(x_train_raw, y_train_raw)
    
    y_pred = model.predict(X_val)
    
    f1_mac = f1_score(y_val_1d, y_pred, average='macro')
    report = classification_report(y_val_1d, y_pred, zero_division=0)
    
    save_confusion_matrix_xgb(y_val_1d, y_pred, trial.number)
    
    log_msg = f"""Trial {trial.number}
Experimento: XgBoost Baseline
Params: {trial.params}

Metricas Clave
F1 Macro: {f1_mac:.4f}

Classification Report:
{report}
"""
    
    with open(f"{dir_name}/Trial_{trial.number}_Metrics.log", 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} finalizado | F1 Macro: {f1_mac:.4f}")
    
    return f1_mac

print("Iniciando búsqueda de Hiperparámetros para Baseline XGBoost...")
study_xgb = optuna.create_study(direction='maximize', study_name="XGBoost_Baseline_Optimization")

study_xgb.optimize(objective_xgb, n_trials=100)

print(f"Mejor Trial: {study_xgb.best_trial.number}")
print(f"Mejor F1 Macro: {study_xgb.best_value:.4f}")
print(f"Mejores parametros:\n{study_xgb.best_params}")

with open(f"{dir_name}/Resumen_Final_Baseline.txt", 'w', encoding='utf-8') as f:
    f.write(f"Resultados Finales XGBoost Baseline\n")
    f.write(f"Mejor F1 Macro: {study_xgb.best_value:.4f}\n")
    f.write(f"Mejores Hiperparámetros: {study_xgb.best_params}\n")

[I 2026-03-30 18:47:58,410] A new study created in memory with name: XGBoost_Baseline_Optimization


Iniciando búsqueda de Hiperparámetros para Baseline XGBoost...


[I 2026-03-30 18:50:17,995] Trial 0 finished with value: 0.8024822602893297 and parameters: {'n_estimators': 691, 'max_depth': 16, 'learning_rate': 0.03586413493009492, 'subsample': 0.8413324894255321, 'colsample_bytree': 0.8390929541283617, 'colsample_bynode': 0.9636390979686924}. Best is trial 0 with value: 0.8024822602893297.


Trial 0 finalizado | F1 Macro: 0.8025


[I 2026-03-30 18:51:00,385] Trial 1 finished with value: 0.8001832470767306 and parameters: {'n_estimators': 339, 'max_depth': 11, 'learning_rate': 0.055345404066702866, 'subsample': 0.8698085455274814, 'colsample_bytree': 0.9378390705150058, 'colsample_bynode': 0.7615448095260772}. Best is trial 0 with value: 0.8024822602893297.


...

Trial 97 finalizado | F1 Macro: 0.8022


[I 2026-03-30 21:44:35,563] Trial 98 finished with value: 0.8030957763572316 and parameters: {'n_estimators': 318, 'max_depth': 29, 'learning_rate': 0.039591881085740986, 'subsample': 0.8938654460486102, 'colsample_bytree': 0.7781972085950457, 'colsample_bynode': 0.9487350892434344}. Best is trial 47 with value: 0.80474176034557.


Trial 98 finalizado | F1 Macro: 0.8031


[I 2026-03-30 21:47:41,507] Trial 99 finished with value: 0.8031961413775739 and parameters: {'n_estimators': 713, 'max_depth': 27, 'learning_rate': 0.04700615627431451, 'subsample': 0.9330481809577558, 'colsample_bytree': 0.7475856892425728, 'colsample_bynode': 0.7778142627788118}. Best is trial 47 with value: 0.80474176034557.


Trial 99 finalizado | F1 Macro: 0.8032
Mejor Trial: 47
Mejor F1 Macro: 0.8047
Mejores parametros:
{'n_estimators': 533, 'max_depth': 22, 'learning_rate': 0.07198458132074173, 'subsample': 0.7493272281145942, 'colsample_bytree': 0.7898260520656288, 'colsample_bynode': 0.9225913983436678}


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import xgboost as xgb
import optuna

dir_name = "Logs_XGBoost_Baseline_2"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix_xgb_exp2(y_true, y_pred, trial_number, dataset_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM - {dataset_name.upper()} - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/{dataset_name}_Trial_{trial_number}_XGB_{phase}_CM.png')
    plt.close()

datasets_to_evaluate = ['smote', 'smote_enn', 'smote_tomek']

for dataset_name in datasets_to_evaluate:
    print(f"Xgboost - Experimento 2: Dataset {dataset_name.upper()}")
    
    X_train_current, y_train_current = train_datasets[dataset_name]
    
    def objective_xgb_exp2(trial):
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 700),
            'max_depth': trial.suggest_int('max_depth', 6, 30),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'colsample_bynode': trial.suggest_float('colsample_bynode', 0.7, 1.0),
            'tree_method': 'hist', 
            'device': 'cuda',
            'objective': 'multi:softmax',
            'num_class': 15,
            'random_state': 42,
            'n_jobs': -1
        }
        
        model = xgb.XGBClassifier(**xgb_params)
        model.fit(X_train_current, y_train_current)
        
        y_pred = model.predict(X_val)
        
        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        
        save_confusion_matrix_xgb_exp2(y_val_1d, y_pred, trial.number, dataset_name, phase="Val")
        
        log_msg = f"""Trial {trial.number}
Experimento 2 XGB: Oversampling ({dataset_name})
Params: {trial.params}

F1 Macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{dir_name}/{dataset_name}_Trial_{trial.number}_Metrics.log", 'w') as f:
            f.write(log_msg)
            
        print(f"[{dataset_name.upper()}] Trial {trial.number} finalizado | F1 Macro: {f1_mac:.4f}")
        return f1_mac

    study_xgb = optuna.create_study(direction='maximize', study_name=f"XGB_Exp2_{dataset_name}")
    study_xgb.optimize(objective_xgb_exp2, n_trials=50) 
    
    print(f"\nResultados para: {dataset_name.upper()}")
    print(f"Mejor Trial: {study_xgb.best_trial.number}")
    print(f"Mejor F1 Macro: {study_xgb.best_value:.4f}")
    
    with open(f"{dir_name}/Resumen_Exp2_Oversampling.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Dataset: {dataset_name.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_xgb.best_value:.4f} (Trial {study_xgb.best_trial.number})\n")
        res_file.write(f"Params: {study_xgb.best_params}\n\n")

[I 2026-03-31 05:10:35,773] A new study created in memory with name: XGB_Exp2_smote


Xgboost - Experimento 2: Dataset SMOTE


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [05:15:13] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
[I 2026-03-31 05:15:14,053] Trial 0 finished with value: 0.8054174558638898 and parameters: {'n_estimators': 457, 'max_depth': 30, 'learning_rate': 0.010243332015896884, 'subsample': 0.9439840240815742, 'colsample_bytree': 0.7947637916289167, 'colsample_bynode': 0.9909988024831515}. Best is trial 0 with value: 0.8054174558638898.


[SMOTE] Trial 0 finalizado | F1 Macro: 0.8054


[I 2026-03-31 05:17:07,707] Trial 1 finished with value: 0.8148293998071559 and parameters: {'n_estimators': 236, 'max_depth': 24, 'learning_rate': 0.04429824686813942, 'subsample': 0.8079938315051713, 'colsample_bytree': 0.8271237061897919, 'colsample_bynode': 0.8367140835359969}. Best is trial 1 with value: 0.8148293998071559.


...

[SMOTE_TOMEK] Trial 47 finalizado | F1 Macro: 0.7997


[I 2026-03-31 10:06:44,589] Trial 48 finished with value: 0.8235427441003625 and parameters: {'n_estimators': 443, 'max_depth': 26, 'learning_rate': 0.03065450772209309, 'subsample': 0.7170352497722854, 'colsample_bytree': 0.8415367027572737, 'colsample_bynode': 0.8649914096594061}. Best is trial 48 with value: 0.8235427441003625.


[SMOTE_TOMEK] Trial 48 finalizado | F1 Macro: 0.8235


[I 2026-03-31 10:09:54,880] Trial 49 finished with value: 0.8131390155428203 and parameters: {'n_estimators': 370, 'max_depth': 29, 'learning_rate': 0.01928246503022425, 'subsample': 0.7154274371930568, 'colsample_bytree': 0.8412686580721138, 'colsample_bynode': 0.8901765660515525}. Best is trial 48 with value: 0.8235427441003625.


[SMOTE_TOMEK] Trial 49 finalizado | F1 Macro: 0.8131

Resultados para: SMOTE_TOMEK
Mejor Trial: 48
Mejor F1 Macro: 0.8235


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import xgboost as xgb
import optuna

dir_name = "Logs_XGBoost_Baseline_3"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix_xgb_exp3(y_true, y_pred, trial_number, method_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM - {method_name.upper()} - Trial {trial_number}')
    plt.savefig(f'{dir_name}/{method_name}_Trial_{trial_number}_XGB_CM.png')
    plt.close()

X_train_raw, y_train_raw = train_datasets['none']
weight_methods = ['balanced', 'polynomial', 'effective_samples', 'focal']

for method_name in weight_methods:
    print(f"XGBoost - Experimento 3: Pesos {method_name.upper()}")
    
    def objective_xgb_exp3(trial):
        if method_name == 'balanced':
            weight_dict = balanced_class_weight(y_train_raw)
        elif method_name == 'polynomial':
            alpha = trial.suggest_float('poly_alpha', 0.1, 1.0)
            weight_dict = polynomial_class_weight(y_train_raw, alpha=alpha)
        elif method_name == 'effective_samples':
            beta = trial.suggest_float('beta_eff_samp', 0.9, 0.9999, log=True)
            weight_dict = effective_samples_class_weight(y_train_raw, beta=beta)
        elif method_name == 'focal':
            gamma = trial.suggest_float('focal_gamma', 0.5, 5.0)
            weight_dict = focal_class_weight_improved(y_train_raw, gamma=gamma)
        
        map_func = np.vectorize(lambda x: weight_dict.get(x, 1.0))
        sample_weights_xgb = map_func(y_train_raw)
        
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 6, 30),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'colsample_bynode': trial.suggest_float('colsample_bynode', 0.7, 1.0),
            'tree_method': 'hist', 
            'device': 'cuda',
            'objective': 'multi:softmax',
            'num_class': 15,
            'random_state': 42,
            'n_jobs': -1
        }
        
        model = xgb.XGBClassifier(**xgb_params)
        model.fit(X_train_raw, y_train_raw, sample_weight=sample_weights_xgb)
        
        y_pred = model.predict(X_val)
        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        
        save_confusion_matrix_xgb_exp3(y_val_1d, y_pred, trial.number, method_name)
        
        log_msg = f"Trial {trial.number} | {method_name}\nParams: {trial.params}\nF1: {f1_mac:.4f}\n{report}"
        with open(f"{dir_name}/{method_name}_Trial_{trial.number}_Metrics.log", 'w') as f:
            f.write(log_msg)
            
        print(f"[{method_name.upper()}] Trial {trial.number} | F1: {f1_mac:.4f}")
        return f1_mac

    study_xgb = optuna.create_study(direction='maximize', study_name=f"XGB_Exp3_{method_name}")
    study_xgb.optimize(objective_xgb_exp3, n_trials=50) 
    
    with open(f"{dir_name}/Resumen_Exp3_Weights.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Método de Peso: {method_name.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_xgb.best_value:.4f} (Trial {study_xgb.best_trial.number})\n")
        res_file.write(f"Params: {study_xgb.best_params}\n\n")

[I 2026-03-31 10:09:54,898] A new study created in memory with name: XGB_Exp3_balanced


XGBoost - Experimento 3: Pesos BALANCED


[I 2026-03-31 10:11:21,951] Trial 0 finished with value: 0.8560256559524154 and parameters: {'n_estimators': 784, 'max_depth': 9, 'learning_rate': 0.025558583449770002, 'subsample': 0.9625770766372108, 'colsample_bytree': 0.9291830543918301, 'colsample_bynode': 0.8857274922203116}. Best is trial 0 with value: 0.8560256559524154.


[BALANCED] Trial 0 | F1: 0.8560


[I 2026-03-31 10:13:22,983] Trial 1 finished with value: 0.8636338115653752 and parameters: {'n_estimators': 734, 'max_depth': 13, 'learning_rate': 0.02748270624815257, 'subsample': 0.8026563207863767, 'colsample_bytree': 0.9844432544769547, 'colsample_bynode': 0.7972248314153842}. Best is trial 1 with value: 0.8636338115653752.


...

[FOCAL] Trial 47 | F1: 0.4763


[I 2026-03-31 14:15:58,366] Trial 48 finished with value: 0.2665927708819687 and parameters: {'focal_gamma': 4.572970154931653, 'n_estimators': 999, 'max_depth': 21, 'learning_rate': 0.08125856874393565, 'subsample': 0.8339282657705146, 'colsample_bytree': 0.9522632142714611, 'colsample_bynode': 0.9064710016598923}. Best is trial 23 with value: 0.6650778169187956.


[FOCAL] Trial 48 | F1: 0.2666


[I 2026-03-31 14:16:44,188] Trial 49 finished with value: 0.551239272228317 and parameters: {'focal_gamma': 1.1382568585977801, 'n_estimators': 856, 'max_depth': 25, 'learning_rate': 0.024274109491164684, 'subsample': 0.972182928352109, 'colsample_bytree': 0.8179394630339704, 'colsample_bynode': 0.7908395375256784}. Best is trial 23 with value: 0.6650778169187956.


[FOCAL] Trial 49 | F1: 0.5512


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import xgboost as xgb
import optuna

dir_name = "Logs_XGBoost_Baseline_4"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix_xgb_exp4(y_true, y_pred, trial_number, method_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM - FS: {method_name.upper()} - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/{method_name}_Trial_{trial_number}_XGB_CM.png')
    plt.close()

X_train_raw, y_train_raw = train_datasets['none']
total_columns = X_train_raw.shape[1]

fs_methods = ['f_classif', 'tree']

for method in fs_methods:
    print(f"XGBoost - Experimento 4: Selección {method.upper()}")
    
    def objective_xgb_exp4(trial):
        k_features = trial.suggest_int('k_features', 30, total_columns)
        
        selector = FeatureSelector(method=method, k=k_features)
        X_train_fs = selector.fit_transform(X_train_raw, y_train_raw)
        X_val_fs = selector.transform(X_val)
        
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 6, 30),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'colsample_bynode': trial.suggest_float('colsample_bynode', 0.7, 1.0),
            'tree_method': 'hist', 
            'device': 'cuda',
            'objective': 'multi:softmax',
            'num_class': 15,
            'random_state': 42,
            'n_jobs': -1
        }
        
        model = xgb.XGBClassifier(**xgb_params)
        model.fit(X_train_fs, y_train_raw)
        
        y_pred = model.predict(X_val_fs)
        
        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        save_confusion_matrix_xgb_exp4(y_val_1d, y_pred, trial.number, method)
        
        log_msg = f"""Trial {trial.number}
Experimento 4 XGB: Feature Selection ({method})
Params FS (k): {k_features}
Params XGB: {trial.params}

F1 Macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{dir_name}/{method}_Trial_{trial.number}.log", 'w', encoding='utf-8') as f:
            f.write(log_msg)
            
        print(f"[{method.upper()} | k={k_features}] Trial {trial.number} | F1: {f1_mac:.4f}")
        return f1_mac

    study_xgb = optuna.create_study(direction='maximize', study_name=f"XGB_Exp4_{method}")
    study_xgb.optimize(objective_xgb_exp4, n_trials=50) 
    
    with open(f"{dir_name}/Resumen_Exp4_FS.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"FS Método: {method.upper()}\n")
        res_file.write(f"Mejor F1: {study_xgb.best_value:.4f} (Trial {study_xgb.best_trial.number})\n")
        res_file.write(f"Params: {study_xgb.best_params}\n\n")

[I 2026-03-31 14:16:44,206] A new study created in memory with name: XGB_Exp4_f_classif


XGBoost - Experimento 4: Selección F_CLASSIF


[I 2026-03-31 14:17:48,661] Trial 0 finished with value: 0.7098725129194765 and parameters: {'k_features': 31, 'n_estimators': 397, 'max_depth': 13, 'learning_rate': 0.019423996463316256, 'subsample': 0.7583019751037764, 'colsample_bytree': 0.7975178111871899, 'colsample_bynode': 0.9076643641046664}. Best is trial 0 with value: 0.7098725129194765.


[F_CLASSIF | k=31] Trial 0 | F1: 0.7099


[I 2026-03-31 14:19:08,945] Trial 1 finished with value: 0.7656110368895855 and parameters: {'k_features': 60, 'n_estimators': 299, 'max_depth': 26, 'learning_rate': 0.019790074531790294, 'subsample': 0.961925720030087, 'colsample_bytree': 0.8084431577139108, 'colsample_bynode': 0.7340004137934645}. Best is trial 1 with value: 0.7656110368895855.


...

[TREE | k=61] Trial 47 | F1: 0.8016


[I 2026-03-31 17:12:38,467] Trial 48 finished with value: 0.7750798691711073 and parameters: {'k_features': 30, 'n_estimators': 675, 'max_depth': 24, 'learning_rate': 0.049505610025508316, 'subsample': 0.8158273175373052, 'colsample_bytree': 0.8704882353665906, 'colsample_bynode': 0.9437580404882872}. Best is trial 27 with value: 0.8433308716505998.


[TREE | k=30] Trial 48 | F1: 0.7751


[I 2026-03-31 17:13:33,599] Trial 49 finished with value: 0.8071422786370995 and parameters: {'k_features': 34, 'n_estimators': 396, 'max_depth': 12, 'learning_rate': 0.02551982300542238, 'subsample': 0.9126513717508827, 'colsample_bytree': 0.8924390335979111, 'colsample_bynode': 0.9663421539417664}. Best is trial 27 with value: 0.8433308716505998.


[TREE | k=34] Trial 49 | F1: 0.8071


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import xgboost as xgb
import optuna
from scipy.special import softmax


dir_name = "Logs_XGBoost_Baseline_5"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix_xgb_exp5(y_true, y_pred, trial_number, loss_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM - Loss: {loss_name.upper()} - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/{loss_name}_Trial_{trial_number}_XGB_CM.png')
    plt.close()

X_train_raw, y_train_raw = train_datasets['none']

def get_focal_loss_obj(gamma):
    def focal_loss(y_true, y_pred):
        if y_pred.ndim == 1:
            y_pred = y_pred.reshape(y_true.shape[0], -1)
            
        p = np.exp(y_pred - np.max(y_pred, axis=1, keepdims=True))
        p /= np.sum(p, axis=1, keepdims=True)

        y_ohe = np.zeros_like(p)
        for i in range(len(y_true)):
            y_ohe[i, int(y_true[i])] = 1.0

        p_t = p * y_ohe + (1 - p) * (1 - y_ohe)
        mod_factor = np.power(1.0 - p_t, gamma)

        grad = (p - y_ohe) * mod_factor
        hess = p * (1.0 - p) * mod_factor

        return grad, hess
    return focal_loss

loss_functions = ['cross_entropy', 'focal_loss']

for loss_name in loss_functions:
    print(f"XGBoost - Experimento 5: Función de Pérdida {loss_name.upper()}")
    
    def objective_xgb_exp5(trial):
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 6, 15),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'colsample_bynode': trial.suggest_float('colsample_bynode', 0.7, 1.0),
            'tree_method': 'hist', 
            'device': 'cuda',
            'num_class': 15,
            'random_state': 42,
            'n_jobs': -1
        }

        if loss_name == 'cross_entropy':
            xgb_params['objective'] = 'multi:softmax'
            model = xgb.XGBClassifier(**xgb_params)
            model.fit(X_train_raw, y_train_raw)
            y_pred = model.predict(X_val)
            
        elif loss_name == 'focal_loss':
            gamma_val = trial.suggest_float('gamma', 0.5, 5.0)
            
            custom_obj = get_focal_loss_obj(gamma=gamma_val)
            
            xgb_params['objective'] = custom_obj 
            
            model = xgb.XGBClassifier(**xgb_params)
            
            model.fit(X_train_raw, y_train_raw)
            
            y_pred_margins = model.predict(X_val, output_margin=True)
            
            y_pred_probs = softmax(y_pred_margins, axis=1)
            y_pred = np.argmax(y_pred_probs, axis=1)

        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        save_confusion_matrix_xgb_exp5(y_val_1d, y_pred, trial.number, loss_name)
        
        log_msg = f"""Trial {trial.number}
Experimento 5 XGB: Loss Function ({loss_name})
Params XGB: {trial.params}

F1 Macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{dir_name}/{loss_name}_Trial_{trial.number}.log", 'w', encoding='utf-8') as f:
            f.write(log_msg)
            
        print(f"[{loss_name.upper()}] Trial {trial.number} | F1: {f1_mac:.4f}")
        return f1_mac

    study_xgb = optuna.create_study(direction='maximize', study_name=f"XGB_Exp5_{loss_name}")
    study_xgb.optimize(objective_xgb_exp5, n_trials=50) 
    
    with open(f"{dir_name}/Resumen_Exp5_Loss.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Función de Pérdida: {loss_name.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_xgb.best_value:.4f} (Trial {study_xgb.best_trial.number})\n")
        res_file.write(f"Params Ganadores: {study_xgb.best_params}\n\n")

print("\nExperimento 5 de Funciones de Pérdida completado")

[I 2026-03-31 18:43:48,884] A new study created in memory with name: XGB_Exp5_cross_entropy


XGBoost - Experimento 5: Función de Pérdida CROSS_ENTROPY


[I 2026-03-31 18:45:06,226] Trial 0 finished with value: 0.8019602848781953 and parameters: {'n_estimators': 425, 'max_depth': 14, 'learning_rate': 0.10497398522143668, 'subsample': 0.820002156906197, 'colsample_bytree': 0.73493291941823, 'colsample_bynode': 0.9992363362925866}. Best is trial 0 with value: 0.8019602848781953.


[CROSS_ENTROPY] Trial 0 | F1: 0.8020


[I 2026-03-31 18:45:31,902] Trial 1 finished with value: 0.8006622873530548 and parameters: {'n_estimators': 315, 'max_depth': 7, 'learning_rate': 0.09550443478684159, 'subsample': 0.9353763002388545, 'colsample_bytree': 0.7090639127687836, 'colsample_bynode': 0.7126180132667869}. Best is trial 0 with value: 0.8019602848781953.


...

[FOCAL_LOSS] Trial 47 | F1: 0.7832


[I 2026-04-01 01:12:51,018] Trial 48 finished with value: 0.6911444640740005 and parameters: {'n_estimators': 451, 'max_depth': 9, 'learning_rate': 0.03350915512504154, 'subsample': 0.8296023496137941, 'colsample_bytree': 0.8384553723149899, 'colsample_bynode': 0.7799326786882859, 'gamma': 3.511834822490949}. Best is trial 43 with value: 0.8035152755257976.


[FOCAL_LOSS] Trial 48 | F1: 0.6911


[I 2026-04-01 01:20:03,284] Trial 49 finished with value: 0.7903091372041009 and parameters: {'n_estimators': 401, 'max_depth': 10, 'learning_rate': 0.14941364525139592, 'subsample': 0.9516599716238922, 'colsample_bytree': 0.7595897688395923, 'colsample_bynode': 0.8332628854059816, 'gamma': 1.3000752521709829}. Best is trial 43 with value: 0.8035152755257976.


[FOCAL_LOSS] Trial 49 | F1: 0.7903

Experimento 5 de Funciones de Pérdida completado


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import xgboost as xgb
import optuna

dir_name = "Logs_XGBoost_Baseline_6"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix_final(y_true, y_pred, trial_number, ds_name, fs_name, w_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM Final XGB - Trial {trial_number}\nDS: {ds_name} | FS: {fs_name} | W: {w_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/Trial_{trial_number}_Final_CM.png')
    plt.close()

def objective_final_xgb(trial):
    ds_name = trial.suggest_categorical('dataset', ['none', 'smote_tomek'])
    X_train_raw, y_train_raw = train_datasets[ds_name]
    total_cols = X_train_raw.shape[1]

    fs_method = trial.suggest_categorical('fs_method', ['none', 'f_classif'])
    if fs_method == 'f_classif':
        k_opt = trial.suggest_int('k_features', 30, total_cols)
        selector = FeatureSelector(method='f_classif', k=k_opt)
        X_train_final = selector.fit_transform(X_train_raw, y_train_raw)
        X_val_final = selector.transform(X_val)
    else:
        k_opt = total_cols
        X_train_final = X_train_raw
        X_val_final = X_val

    w_method = trial.suggest_categorical('weight_method', ['none', 'balanced'])
    if w_method == 'balanced':
        weight_dict = balanced_class_weight(y_train_raw)
        sample_weights_xgb = np.vectorize(lambda x: weight_dict.get(x, 1.0))(y_train_raw)
    else:
        sample_weights_xgb = None

    xgb_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 6, 26), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.6, 1.0),
        'tree_method': 'hist', 
        'device': 'cuda',
        'objective': 'multi:softmax',
        'num_class': 15,
        'random_state': 42,
        'n_jobs': -1
    }

    model = xgb.XGBClassifier(**xgb_params)
    
    if sample_weights_xgb is not None:
        model.fit(X_train_final, y_train_raw, sample_weight=sample_weights_xgb)
    else:
        model.fit(X_train_final, y_train_raw)
        
    y_pred = model.predict(X_val_final)
    
    f1_mac = f1_score(y_val_1d, y_pred, average='macro')
    report = classification_report(y_val_1d, y_pred, zero_division=0)
    
    save_confusion_matrix_final(y_val_1d, y_pred, trial.number, ds_name, fs_method, w_method)
    
    log_msg = f"""Trial {trial.number} - XGBoost Experimento 6
Arquitectura: DS={ds_name} | FS={fs_method} (k={k_opt}) | W={w_method}
Params XGB: {trial.params}

F1 Macro: {f1_mac:.4f}
{report}
"""
    with open(f"{dir_name}/Trial_{trial.number}_Metrics.log", 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} | F1: {f1_mac:.4f} | DS: {ds_name} | FS: {fs_method} | W: {w_method}")
    return f1_mac

study_xgb = optuna.create_study(direction='maximize', study_name=f"XGB_Exp6_Final")
study_xgb.optimize(objective_final_xgb, n_trials=100) 

print(f"Mejor F1 Macro: {study_xgb.best_value:.4f}")
print(f"Mejor Configuración: {study_xgb.best_params}")
    
with open(f"{dir_name}/Resumen_Exp6.txt", 'a', encoding='utf-8') as res_file:
    res_file.write("Resumen Experimento 6")
    res_file.write(f"Mejor F1 Macro: {study_xgb.best_value:.4f} (Trial {study_xgb.best_trial.number})\n")
    res_file.write(f"Params Ganadores: {study_xgb.best_params}\n")

print("\nExperimento 6 completado")

[I 2026-04-01 02:42:17,891] A new study created in memory with name: XGB_Exp6_Final
[I 2026-04-01 02:42:43,664] Trial 0 finished with value: 0.6150793420258432 and parameters: {'dataset': 'smote_tomek', 'fs_method': 'f_classif', 'k_features': 48, 'weight_method': 'balanced', 'n_estimators': 257, 'max_depth': 6, 'learning_rate': 0.07730667989987955, 'subsample': 0.6443606612237299, 'colsample_bytree': 0.6984124831124311, 'colsample_bynode': 0.6844970919447632}. Best is trial 0 with value: 0.6150793420258432.


Trial 0 | F1: 0.6151 | DS: smote_tomek | FS: f_classif | W: balanced


[I 2026-04-01 02:43:31,662] Trial 1 finished with value: 0.8513672888284858 and parameters: {'dataset': 'none', 'fs_method': 'f_classif', 'k_features': 46, 'weight_method': 'balanced', 'n_estimators': 288, 'max_depth': 15, 'learning_rate': 0.19744602678721482, 'subsample': 0.970337341424778, 'colsample_bytree': 0.7262036231638203, 'colsample_bynode': 0.9618707462170799}. Best is trial 1 with value: 0.8513672888284858.


Trial 1 | F1: 0.8514 | DS: none | FS: f_classif | W: balanced


[I 2026-04-01 02:44:54,464] Trial 2 finished with value: 0.8214888615270035 and parameters: {'dataset': 'smote_tomek', 'fs_method': 'none', 'weight_method': 'balanced', 'n_estimators': 235, 'max_depth': 16, 'learning_rate': 0.015548992071722221, 'subsample': 0.9503151926210962, 'colsample_bytree': 0.6011925888409347, 'colsample_bynode': 0.9617527140528104}. Best is trial 1 with value: 0.8513672888284858.


...

Trial 97 | F1: 0.8623 | DS: none | FS: none | W: balanced


[I 2026-04-01 05:19:49,852] Trial 98 finished with value: 0.8026748979032753 and parameters: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 670, 'max_depth': 11, 'learning_rate': 0.08483087030487012, 'subsample': 0.9049972446782465, 'colsample_bytree': 0.715409264563494, 'colsample_bynode': 0.9484816331212067}. Best is trial 75 with value: 0.8735309183984922.


Trial 98 | F1: 0.8027 | DS: none | FS: none | W: none


[I 2026-04-01 05:21:06,506] Trial 99 finished with value: 0.8521708980672088 and parameters: {'dataset': 'none', 'fs_method': 'f_classif', 'k_features': 51, 'weight_method': 'balanced', 'n_estimators': 611, 'max_depth': 12, 'learning_rate': 0.1251880149795541, 'subsample': 0.9865639872162701, 'colsample_bytree': 0.6809890713593878, 'colsample_bynode': 0.9913653078835856}. Best is trial 75 with value: 0.8735309183984922.


Trial 99 | F1: 0.8522 | DS: none | FS: f_classif | W: balanced
Mejor F1 Macro: 0.8735
Mejor Configuración: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'balanced', 'n_estimators': 783, 'max_depth': 13, 'learning_rate': 0.11687847972600902, 'subsample': 0.9302273616375747, 'colsample_bytree': 0.6661492479334324, 'colsample_bynode': 0.9386104663924604}

Experimento 6 completado


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import xgboost as xgb
import optuna
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")
dir_name = "Logs_XGBoost_Baseline_7"
os.makedirs(dir_name, exist_ok=True)

def save_confusion_matrix_final(y_true, y_pred, trial_number, ds_name, fs_name, w_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM Final XGB - Trial {trial_number}\nDS: {ds_name} | FS: {fs_name} | W: {w_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{dir_name}/Trial_{trial_number}_Final_CM.png')
    plt.close()

def objective_final_xgb(trial):
    ds_name = trial.suggest_categorical('dataset', ['none', 'smote_tomek'])
    X_train_raw, y_train_raw = train_datasets_grouped[ds_name]
    total_cols = X_train_raw.shape[1]

    fs_method = trial.suggest_categorical('fs_method', ['none', 'f_classif'])
    if fs_method == 'f_classif':
        k_opt = trial.suggest_int('k_features', 30, total_cols)
        selector = FeatureSelector(method='f_classif', k=k_opt)
        X_train_final = selector.fit_transform(X_train_raw, y_train_raw)
        X_val_final = selector.transform(X_val_imp_grouped)
    else:
        k_opt = total_cols
        X_train_final = X_train_raw
        X_val_final = X_val_imp_grouped

    w_method = trial.suggest_categorical('weight_method', ['none', 'balanced'])
    if w_method == 'balanced':
        weight_dict = balanced_class_weight(y_train_raw)
        sample_weights_xgb = np.vectorize(lambda x: weight_dict.get(x, 1.0))(y_train_raw)
    else:
        sample_weights_xgb = None

    xgb_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 6, 26), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.6, 1.0),
        'tree_method': 'hist', 
        'device': 'cuda',
        'objective': 'multi:softmax',
        'num_class': 13,
        'random_state': 42,
        'n_jobs': -1
    }

    model = xgb.XGBClassifier(**xgb_params)
    
    if sample_weights_xgb is not None:
        model.fit(X_train_final, y_train_raw, sample_weight=sample_weights_xgb)
    else:
        model.fit(X_train_final, y_train_raw)
        
    y_pred = model.predict(X_val_final)
    
    f1_mac = f1_score(y_val_grouped, y_pred, average='macro')
    report = classification_report(y_val_grouped, y_pred, zero_division=0)
    
    save_confusion_matrix_final(y_val_grouped, y_pred, trial.number, ds_name, fs_method, w_method)
    
    log_msg = f"""Trial {trial.number} - XGBoost Experimento 7
Arquitectura: DS={ds_name} | FS={fs_method} (k={k_opt}) | W={w_method}
Params XGB: {trial.params}

F1 Macro: {f1_mac:.4f}
{report}
"""
    with open(f"{dir_name}/Trial_{trial.number}_Metrics.log", 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} | F1: {f1_mac:.4f} | DS: {ds_name} | FS: {fs_method} | W: {w_method}")
    return f1_mac

study_xgb = optuna.create_study(direction='maximize', study_name=f"XGB_Exp7_Final_Grouped_Data")
study_xgb.optimize(objective_final_xgb, n_trials=100) 

print(f"Mejor F1 Macro: {study_xgb.best_value:.4f}")
print(f"Mejor Configuración: {study_xgb.best_params}")
    
with open(f"{dir_name}/Resumen_Exp7.txt", 'a', encoding='utf-8') as res_file:
    res_file.write("Resumen Experimento 7-Datos Agrupados")
    res_file.write(f"Mejor F1 Macro: {study_xgb.best_value:.4f} (Trial {study_xgb.best_trial.number})\n")
    res_file.write(f"Params Ganadores: {study_xgb.best_params}\n")

print("\nExperimento 7 completado")

[I 2026-04-01 23:18:22,515] A new study created in memory with name: XGB_Exp7_Final_Grouped_Data
[I 2026-04-01 23:19:45,120] Trial 0 finished with value: 0.9183426641101737 and parameters: {'dataset': 'smote_tomek', 'fs_method': 'none', 'weight_method': 'balanced', 'n_estimators': 606, 'max_depth': 11, 'learning_rate': 0.010015516798455948, 'subsample': 0.6817485890633337, 'colsample_bytree': 0.7083300139710083, 'colsample_bynode': 0.9329649651389841}. Best is trial 0 with value: 0.9183426641101737.


Trial 0 | F1: 0.9183 | DS: smote_tomek | FS: none | W: balanced


[I 2026-04-01 23:20:11,092] Trial 1 finished with value: 0.923390173600857 and parameters: {'dataset': 'smote_tomek', 'fs_method': 'f_classif', 'k_features': 56, 'weight_method': 'balanced', 'n_estimators': 112, 'max_depth': 18, 'learning_rate': 0.02148852176758552, 'subsample': 0.9584309273303473, 'colsample_bytree': 0.9415141908055006, 'colsample_bynode': 0.650673312835378}. Best is trial 1 with value: 0.923390173600857.


Trial 1 | F1: 0.9234 | DS: smote_tomek | FS: f_classif | W: balanced


[I 2026-04-01 23:21:22,745] Trial 2 finished with value: 0.9276542405895523 and parameters: {'dataset': 'smote_tomek', 'fs_method': 'none', 'weight_method': 'balanced', 'n_estimators': 466, 'max_depth': 14, 'learning_rate': 0.08595230069976173, 'subsample': 0.7384356270250841, 'colsample_bytree': 0.7445492941236099, 'colsample_bynode': 0.7216077351149346}. Best is trial 2 with value: 0.9276542405895523.


...

Trial 97 | F1: 0.9599 | DS: none | FS: f_classif | W: none


[I 2026-04-02 01:28:17,115] Trial 98 finished with value: 0.9618130909439316 and parameters: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 525, 'max_depth': 6, 'learning_rate': 0.18499879781578069, 'subsample': 0.96350627411532, 'colsample_bytree': 0.643822719473229, 'colsample_bynode': 0.7426079610710496}. Best is trial 98 with value: 0.9618130909439316.


Trial 98 | F1: 0.9618 | DS: none | FS: none | W: none


[I 2026-04-02 01:28:50,503] Trial 99 finished with value: 0.9603012411907988 and parameters: {'dataset': 'none', 'fs_method': 'f_classif', 'k_features': 55, 'weight_method': 'none', 'n_estimators': 521, 'max_depth': 6, 'learning_rate': 0.17854072538173646, 'subsample': 0.886002922570473, 'colsample_bytree': 0.7057037883605166, 'colsample_bynode': 0.7709549639350182}. Best is trial 98 with value: 0.9618130909439316.


Trial 99 | F1: 0.9603 | DS: none | FS: f_classif | W: none
Mejor F1 Macro: 0.9618
Mejor Configuración: {'dataset': 'none', 'fs_method': 'none', 'weight_method': 'none', 'n_estimators': 525, 'max_depth': 6, 'learning_rate': 0.18499879781578069, 'subsample': 0.96350627411532, 'colsample_bytree': 0.643822719473229, 'colsample_bynode': 0.7426079610710496}

Experimento 7 completado


In [14]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

log_folder_test_xgb = "Logs_Resultados_Test_Final_XGB"
os.makedirs(log_folder_test_xgb, exist_ok=True)

x_train_none, y_train_none = train_datasets['none']

best_params_xgb_baseline = {
    'n_estimators': 533, 
    'max_depth': 22, 
    'learning_rate': 0.07198458132074173, 
    'subsample': 0.7493272281145942, 
    'colsample_bytree': 0.7898260520656288, 
    'colsample_bynode': 0.9225913983436678,
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42
}

print("Entrenando el mejor modelo XGBoost del Experimento 1...")
final_xgb_baseline = xgb.XGBClassifier(**best_params_xgb_baseline)
final_xgb_baseline.fit(x_train_none, y_train_none)

print("Generando predicciones en el Test Set...")
preds_test_xgb = final_xgb_baseline.predict(X_test)

f1_mac_test = f1_score(y_test_1d, preds_test_xgb, average='macro')
report_test = classification_report(y_test_1d, preds_test_xgb, zero_division=0)

print(f"\nResultados finales en Test - XGBoost Baseline")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_xgb)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - XGBoost Baseline\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_xgb}/CM_Test_XGB_Baseline_Exp1.png')
plt.close()

with open(f"{log_folder_test_xgb}/Reporte_Test_XGB_Baseline_Test_1.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - XGBoost Baseline\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_xgb_baseline}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente. Archivos guardados en '{log_folder_test_xgb}'")

Entrenando el mejor modelo XGBoost del Experimento 1...


Generando predicciones en el Test Set...

Resultados finales en Test - XGBoost Baseline
F1 Macro Alcanzado: 0.8321

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.77      0.39      0.52       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      1.00      1.00       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      1.00     23840
          11       0.99      1.00      0.99       884
          12       0.72      0.71      0.71       226
          13       0.00      0.00      0.00         3
          14       0.38      0.40      0.39    

In [15]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix

log_folder_test_xgb = "Logs_Resultados_Test_Final_XGB"
os.makedirs(log_folder_test_xgb, exist_ok=True)

print("Preparando datos de entrenamiento (SMOTE_TOMEK)...")
x_train_smote_tomek, y_train_smote_tomek = train_datasets['smote_tomek']

best_params_xgb_exp2 = {
    'n_estimators': 443, 
    'max_depth': 26, 
    'learning_rate': 0.03065450772209309, 
    'subsample': 0.7170352497722854, 
    'colsample_bytree': 0.8415367027572737, 
    'colsample_bynode': 0.8649914096594061,
    'tree_method': 'hist', 
    'device': 'cuda',
    'random_state': 42
}

print("Entrenando el modelo XGBoost definitivo (Experimento 2 - SMOTE_TOMEK)...")
final_xgb_exp2 = xgb.XGBClassifier(**best_params_xgb_exp2)
final_xgb_exp2.fit(x_train_smote_tomek, y_train_smote_tomek)

print("Generando predicciones en el Test Set...")
preds_test_xgb_exp2 = final_xgb_exp2.predict(X_test)

f1_mac_test = f1_score(y_test_1d, preds_test_xgb_exp2, average='macro')
report_test = classification_report(y_test_1d, preds_test_xgb_exp2, zero_division=0)

print(f"\nResultados finales en Test - XGBoost Experimento 2 (SMOTE_TOMEK)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_xgb_exp2)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - XGBoost Exp 2 (SMOTE_TOMEK)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_xgb}/CM_Test_XGB_Baseline_Exp2.png')
plt.close()

with open(f"{log_folder_test_xgb}/Reporte_Test_XGB_Baseline_Test_2.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - XGBoost Experimento 2 (SMOTE_TOMEK)\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_xgb_exp2}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente para experimento 2. Archivos guardados en '{log_folder_test_xgb}'")

Preparando datos de entrenamiento (SMOTE_TOMEK)...
Entrenando el modelo XGBoost definitivo (Experimento 2 - SMOTE_TOMEK)...
Generando predicciones en el Test Set...

Resultados finales en Test - XGBoost Experimento 2 (SMOTE_TOMEK)
F1 Macro Alcanzado: 0.8348

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.38      0.74      0.50       295
           2       1.00      1.00      1.00     19204
           3       0.99      1.00      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       0.99      1.00      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      1.00     23840
          11       0.98      1.00      0.99       884
          12

In [16]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

log_folder_test_xgb = "Logs_Resultados_Test_Final_XGB"
os.makedirs(log_folder_test_xgb, exist_ok=True)

print("Preparando datos de entrenamiento...")
x_train_none, y_train_none = train_datasets['none']

best_params_xgb_exp3 = {
    'n_estimators': 882, 
    'max_depth': 24, 
    'learning_rate': 0.14866856132426257, 
    'subsample': 0.8310515115996882, 
    'colsample_bytree': 0.8034948789651247, 
    'colsample_bynode': 0.9011915856068373,
    'tree_method': 'hist', 
    'device': 'cuda',
    'random_state': 42
}

print("Calculando vector de pesos balanceados (sample_weight)...")
pesos_muestra = compute_sample_weight('balanced', y_train_none)

print("Entrenando el modelo XGBoost (Experimento 3 - BALANCED)...")
final_xgb_exp3 = xgb.XGBClassifier(**best_params_xgb_exp3)

final_xgb_exp3.fit(x_train_none, y_train_none, sample_weight=pesos_muestra)

print("Generando predicciones en el Test Set...")
preds_test_xgb_exp3 = final_xgb_exp3.predict(X_test)

f1_mac_test = f1_score(y_test_1d, preds_test_xgb_exp3, average='macro')
report_test = classification_report(y_test_1d, preds_test_xgb_exp3, zero_division=0)

print(f"\nResultados finales en Test - XGBoost Experimento 3 (BALANCED)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_xgb_exp3)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - XGBoost Exp 3 (BALANCED)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_xgb}/CM_Test_XGB_Baseline_Exp3.png')
plt.close()

with open(f"{log_folder_test_xgb}/Reporte_Test_XGB_Baseline_Test_3.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - XGBoost Experimento 3 (BALANCED)\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_xgb_exp3}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente para experimento 3. Archivos guardados en '{log_folder_test_xgb}'")

Preparando datos de entrenamiento...
Calculando vector de pesos balanceados (sample_weight)...
Entrenando el modelo XGBoost (Experimento 3 - BALANCED)...
Generando predicciones en el Test Set...

Resultados finales en Test - XGBoost Experimento 3 (BALANCED)
F1 Macro Alcanzado: 0.8575

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.45      0.71      0.55       295
           2       1.00      1.00      1.00     19204
           3       0.99      1.00      0.99      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       0.99      1.00      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      1.00     23840
          11       0.98      1.00      

In [17]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix

log_folder_test_xgb = "Logs_Resultados_Test_Final_XGB"
os.makedirs(log_folder_test_xgb, exist_ok=True)

x_train_none, y_train_none = train_datasets['none']

k_opt = 50
best_params_xgb_exp4 = {
    'n_estimators': 213, 
    'max_depth': 24, 
    'learning_rate': 0.05222033178604469, 
    'subsample': 0.938668001851516, 
    'colsample_bytree': 0.8770750107647156, 
    'colsample_bynode': 0.9230183814095745,
    'tree_method': 'hist', 
    'device': 'cuda',
    'random_state': 42
}

print("Aplicando Feature Selection (Método: F_CLASSIF, k=50)...")
selector = FeatureSelector(method='f_classif', k=k_opt)
X_train_fs = selector.fit_transform(x_train_none, y_train_none)
X_test_fs = selector.transform(X_test)

print("Entrenando el modelo XGBoost definitivo (Experimento 4 - F_CLASSIF)...")
final_xgb_exp4 = xgb.XGBClassifier(**best_params_xgb_exp4)
final_xgb_exp4.fit(X_train_fs, y_train_none)

print("Generando predicciones en el Test Set...")
preds_test_xgb_exp4 = final_xgb_exp4.predict(X_test_fs)

f1_mac_test = f1_score(y_test_1d, preds_test_xgb_exp4, average='macro')
report_test = classification_report(y_test_1d, preds_test_xgb_exp4, zero_division=0)

print(f"\nResultados finales en Test - XGBoost Experimento 4 (F_CLASSIF)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_xgb_exp4)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - XGBoost Exp 4 (F_CLASSIF)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_xgb}/CM_Test_XGB_Baseline_Exp4.png')
plt.close()

with open(f"{log_folder_test_xgb}/Reporte_Test_XGB_Baseline_Test_4.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - XGBoost Experimento 4 (F_CLASSIF)\n")
    f.write(f"k_features seleccionadas: {k_opt}\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_xgb_exp4}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente para experimento 4. Archivos guardados en '{log_folder_test_xgb}'")

Aplicando Feature Selection (Método: F_CLASSIF, k=50)...
Entrenando el modelo XGBoost definitivo (Experimento 4 - F_CLASSIF)...
Generando predicciones en el Test Set...

Resultados finales en Test - XGBoost Experimento 4 (F_CLASSIF)
F1 Macro Alcanzado: 0.8286

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.79      0.40      0.53       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      0.99      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      1.00       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      1.00     23840
          11       0.99      1.00      0.99       884
          

In [18]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy.special import softmax

log_folder_test_xgb = "Logs_Resultados_Test_Final_XGB"
os.makedirs(log_folder_test_xgb, exist_ok=True)

def get_focal_loss_obj(gamma):
    def focal_loss(y_true, y_pred):
        if y_pred.ndim == 1:
            y_pred = y_pred.reshape(y_true.shape[0], -1)
            
        p = np.exp(y_pred - np.max(y_pred, axis=1, keepdims=True))
        p /= np.sum(p, axis=1, keepdims=True)

        y_ohe = np.zeros_like(p)
        for i in range(len(y_true)):
            y_ohe[i, int(y_true[i])] = 1.0

        p_t = p * y_ohe + (1 - p) * (1 - y_ohe)
        mod_factor = np.power(1.0 - p_t, gamma)

        grad = (p - y_ohe) * mod_factor
        hess = p * (1.0 - p) * mod_factor

        return grad, hess
    return focal_loss

print("Preparando datos de entrenamiento (Dataset Original)...")
x_train_none, y_train_none = train_datasets['none']

gamma_ganador = 0.7299323194869802
custom_obj = get_focal_loss_obj(gamma=gamma_ganador)

best_params_xgb_exp5 = {
    'n_estimators': 479, 
    'max_depth': 7, 
    'learning_rate': 0.14439910176278958, 
    'subsample': 0.9830945340771192, 
    'colsample_bytree': 0.9970656024220925, 
    'colsample_bynode': 0.7456600287015384,
    'objective': custom_obj,
    'num_class': 15,
    'tree_method': 'hist', 
    'device': 'cuda',
    'random_state': 42,
    'n_jobs': -1
}

print("Entrenando el modelo XGBoost definitivo (Experimento 5 - FOCAL LOSS)...")
final_xgb_exp5 = xgb.XGBClassifier(**best_params_xgb_exp5)
final_xgb_exp5.fit(x_train_none, y_train_none)

print("Generando predicciones crudas y aplicando Softmax en el Test Set...")
y_pred_margins_test = final_xgb_exp5.predict(X_test, output_margin=True)
y_pred_probs_test = softmax(y_pred_margins_test, axis=1)
preds_test_xgb_exp5 = np.argmax(y_pred_probs_test, axis=1)

f1_mac_test = f1_score(y_test_1d, preds_test_xgb_exp5, average='macro')
report_test = classification_report(y_test_1d, preds_test_xgb_exp5, zero_division=0)

print(f"\nResultados finales en Test - XGBoost Experimento 5 (FOCAL LOSS)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_xgb_exp5)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - XGBoost Exp 5 (FOCAL LOSS)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_xgb}/CM_Test_XGB_Baseline_Exp5.png')
plt.close()

with open(f"{log_folder_test_xgb}/Reporte_Test_XGB_Baseline_Test_5.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - XGBoost Experimento 5 (FOCAL LOSS)\n")
    f.write(f"Gamma optimizado: {gamma_ganador}\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_xgb_exp5}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente para experimento 5. Archivos guardados en '{log_folder_test_xgb}'")

Preparando datos de entrenamiento (Dataset Original)...
Entrenando el modelo XGBoost definitivo (Experimento 5 - FOCAL LOSS)...
Generando predicciones crudas y aplicando Softmax en el Test Set...

Resultados finales en Test - XGBoost Experimento 5 (FOCAL LOSS)
F1 Macro Alcanzado: 0.8334

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.78      0.40      0.53       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      1.00      1.00       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      1.00     23840
          11       0.99      1.00   

In [14]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

log_folder_test_xgb = "Logs_Resultados_Test_Final_XGB"
os.makedirs(log_folder_test_xgb, exist_ok=True)

print("Preparando datos de entrenamiento (Dataset Original para Arquitectura Final)...")
x_train_none, y_train_none = train_datasets['none']

best_params_xgb_exp6 = {
    'n_estimators': 783, 
    'max_depth': 13, 
    'learning_rate': 0.11687847972600902, 
    'subsample': 0.9302273616375747, 
    'colsample_bytree': 0.6661492479334324, 
    'colsample_bynode': 0.9386104663924604,
    'tree_method': 'hist', 
    'device': 'cuda',
    'random_state': 42,
    'n_jobs': -1
}

print("Calculando vector de pesos balanceados (sample_weight)...")
pesos_muestra_exp6 = compute_sample_weight('balanced', y_train_none)

print("Entrenando el modelo XGBoost definitivo (Experimento 6 - Arquitectura Final)...")
final_xgb_exp6 = xgb.XGBClassifier(**best_params_xgb_exp6)

final_xgb_exp6.fit(x_train_none, y_train_none, sample_weight=pesos_muestra_exp6)

print("Generando predicciones en el Test Set...")
preds_test_xgb_exp6 = final_xgb_exp6.predict(X_test)

f1_mac_test = f1_score(y_test_1d, preds_test_xgb_exp6, average='macro')
report_test = classification_report(y_test_1d, preds_test_xgb_exp6, zero_division=0)

print(f"\nResultados finales en Test - XGBoost Experimento 6 (Arquitectura Final)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_1d, preds_test_xgb_exp6)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - XGBoost Exp 6 (Arquitectura Final)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_xgb}/CM_Test_XGB_Baseline_Exp6.png')
plt.close()

with open(f"{log_folder_test_xgb}/Reporte_Test_XGB_Baseline_Test_6.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - XGBoost Experimento 6 (Arquitectura Final)\n")
    f.write(f"Configuración: Dataset='none', FS='none', Weight='balanced'\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_xgb_exp6}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente para experimento 6. Archivos guardados en '{log_folder_test_xgb}'")

Preparando datos de entrenamiento (Dataset Original para Arquitectura Final)...
Calculando vector de pesos balanceados (sample_weight)...
Entrenando el modelo XGBoost definitivo (Experimento 6 - Arquitectura Final)...
Generando predicciones en el Test Set...

Resultados finales en Test - XGBoost Experimento 6 (Arquitectura Final)
F1 Macro Alcanzado: 0.8658

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.42      0.76      0.54       295
           2       1.00      1.00      1.00     19204
           3       0.99      1.00      0.99      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       0.99      1.00      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      1.00      1.00         5
          10       

In [17]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

log_folder_test_xgb = "Logs_Resultados_Test_Final_XGB"
os.makedirs(log_folder_test_xgb, exist_ok=True)

print("Preparando datos de entrenamiento (Dataset Original Agrupado)...")
x_train_none_grp, y_train_none_grp = train_datasets_grouped['none']

y_test_grp_1d = y_test_grouped.to_numpy().ravel() if isinstance(y_test_grouped, pd.DataFrame) else y_test_grouped.to_numpy()

best_params_xgb_exp7 = {
    'n_estimators': 525, 
    'max_depth': 6, 
    'learning_rate': 0.18499879781578069, 
    'subsample': 0.96350627411532, 
    'colsample_bytree': 0.643822719473229, 
    'colsample_bynode': 0.7426079610710496,
    'tree_method': 'hist', 
    'device': 'cuda',
    'random_state': 42,
    'n_jobs': -1
}

print("Entrenando el modelo XGBoost definitivo (Experimento 7 - Datos Agrupados)...")
final_xgb_exp7 = xgb.XGBClassifier(**best_params_xgb_exp7)
final_xgb_exp7.fit(x_train_none_grp, y_train_none_grp)

print("Generando predicciones en el Test Set agrupado...")
preds_test_xgb_exp7 = final_xgb_exp7.predict(X_test_imp_grouped)

f1_mac_test = f1_score(y_test_grp_1d, preds_test_xgb_exp7, average='macro')
report_test = classification_report(y_test_grp_1d, preds_test_xgb_exp7, zero_division=0)

print(f"\nResultados finales en Test - XGBoost Experimento 7 (Datos Agrupados)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_grp_1d, preds_test_xgb_exp7)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - XGBoost Exp 7 (Grouped)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_xgb}/CM_Test_XGB_Baseline_Exp7.png')
plt.close()

with open(f"{log_folder_test_xgb}/Reporte_Test_XGB_Baseline_Test_7.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - XGBoost Experimento 7 (Datos Agrupados)\n")
    f.write(f"Configuración: Dataset='none', FS='none', Weight='none'\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_xgb_exp7}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente para experimento 7. Archivos guardados en '{log_folder_test_xgb}'")

Preparando datos de entrenamiento (Dataset Original Agrupado)...
Entrenando el modelo XGBoost definitivo (Experimento 7 - Datos Agrupados)...
Generando predicciones en el Test Set agrupado...

Resultados finales en Test - XGBoost Experimento 7 (Datos Agrupados)
F1 Macro Alcanzado: 0.9039

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.77      0.45      0.57       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      0.50      0.67         2
           9       1.00      0.40      0.57         5
          10       0.99      1.00      1.00     23840
          11       0.99      1.00  